In [17]:
import numpy as np
import os
import itertools
import logging
import matplotlib.pyplot as plt
import torch
import torch.optim as optim
import torch.nn.functional as F
from argparse import ArgumentParser
from torch.distributions import MultivariateNormal
import sys
sys.path.append("../")
import pymbar
from config import get_cfg_defaults
from nf.flows import *
from nf.models import NormalizingFlowModel
from nf.base import EinsteinCrystal
import nf.utils as util
import random
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
def setup_model(cfg):
    if cfg.dataset.rho is not None:
        B=(cfg.dataset.nparticles/(8*cfg.dataset.rho))**(1/3)
    else:
        B=cfg.dataset.ncellx*cfg.dataset.cell_len/2
    boxlength=2*B
    print("B:",B)
    N=cfg.dataset.nparticles*3
    logging.basicConfig(level=logging.DEBUG)
    logger = logging.getLogger(__name__)  
    if cfg.prior.type=="lattice":
        prior = EinsteinCrystal(cfg.prior.lattice_dir, alpha=cfg.prior.alpha,device=cfg.device)
    elif cfg.prior.type=="normal":
        prior = MultivariateNormal(torch.zeros(N).to(cfg.device), torch.eye(N).to(cfg.device))
    if cfg.flow.type=="RealNVP":
        flows = [eval(cfg.flow.type)(dim=N,hidden_dim=cfg.flow.hidden_dim) for _ in range(cfg.flow.nlayers)]
    elif cfg.flow.type=="NSF_AR":
        flows = [eval(cfg.flow.type)(dim=N, K=cfg.flow.nsplines, B=B,hidden_dim=cfg.flow.hidden_dim) for _ in range(cfg.flow.nlayers)]
    elif cfg.flow.type=="NSF_CL":
        mask= np.repeat([[1],[2],[3],[1,3],[1,2],[2,3]],cfg.flow.nlayers//6+1)
        flows = [eval(cfg.flow.type)(size=cfg.dataset.nparticles,dim=3, K=cfg.flow.nsplines, B=B,hidden_dim=cfg.flow.hidden_dim, mask=mask[i]) for i in range(cfg.flow.nlayers)]
    model = NormalizingFlowModel(prior, flows).to(cfg.device)
    optimizer = optim.Adam(model.parameters(), lr=cfg.train_parameters.learning_rate)
    #scheduler = torch. optim.lr_scheduler.ExponentialLR(optimizer, cfg.train_parameters.lr_scheduler_gamma)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, cfg.train_parameters.max_epochs)
    training_data = util.load_position(cfg.dataset.input_dir).to(cfg.device)
    with open(cfg.output.pos_dir, 'w'):
        pass
    if not(os.path.exists(cfg.output.model_dir)):
        os.mkdir(cfg.output.model_dir)
    return model,optimizer,scheduler,training_data,logger,boxlength

In [14]:
def generated_data(cfg,batchsize,nparticles):
    which_dist=[random.randint(0,1) for _ in range(batchsize*nparticles)].reshape((batchsize,nparticles))
    size1=np.sum(which_dist,axis=0)
    samples1=MultivariateNormal(torch.Tensor([0.5,0.5,0.5]).to(cfg.device), 0.1*torch.eye(3)).sample((size1,))
    samples2=MultivariateNormal(torch.Tensor([-0.5,-0.5,-0.5]).to(cfg.device), 0.05*torch.eye(3)).sample((nsamples-size1,))
    return torch.cat((samples1,samples2),0)

In [15]:
def train(cfg,model,optimizer,scheduler,training_data,logger,boxlength):
    with open("base.xyz", 'w'):
            pass
    losses=[]
    max_logprob=140
    lamb=0.5
    for i in range(cfg.train_parameters.max_epochs):
        optimizer.zero_grad()
        x = util.subsample(training_data,nsamples=cfg.train_parameters.batch_size,device=cfg.device)
        z, prior_logprob, log_det = model(x)
        util.write_coord("base.xyz",z.detach(),cfg.dataset.nparticles,boxlength)
        #x1, log_det1 =model.inverse(z)
        #print("check the map is invertible", torch.linalg.norm(x1-x), torch.linalg.norm(log_det1+log_det))
        logprob = prior_logprob + log_det
        forward_loss=-torch.mean(logprob)
        #if i>4000:
            #loss = forward_loss*lamb + (1-lamb)*reverseKL(cfg,model, cfg.train_parameters.batch_size,boxlength)
        #else:
        loss = forward_loss#-U(x,boxlength)/cfg.dataset.kT
        losses.append(loss.mean().data)
        loss.backward()
        optimizer.step()
        scheduler.step()
        if i % cfg.train_parameters.output_freq == 0:
            logger.info(f"Iter: {i}\t" +
                        f"Loss: {loss.mean().data:.2f}\t" +
                        f"Logprob: {logprob.mean().data:.2f}\t" +
                        f"Prior: {prior_logprob.mean().data:.2f}\t" +
                        f"LogDet: {log_det.mean().data:.2f}")
            samples,_,z = model.sample(1)
            util.write_coord(cfg.output.pos_dir,samples,cfg.dataset.nparticles,boxlength)
            if (i>1200) and (-forward_loss>max_logprob):
                max_logprob=-forward_loss
                torch.save({"model":model.state_dict(),"optim": optimizer.state_dict(),
                            "loss":losses},cfg.output.model_dir+cfg.dataset.name+'%d.pth'% (i//cfg.train_parameters.output_freq))

In [18]:
def read_input(dir):
    cfg = get_cfg_defaults()
    cfg.merge_from_file(dir)
    cfg.freeze()
    print(cfg)
    return cfg

In [26]:
cfg=read_input("input/mixed_gauss.yaml")
model,optimizer,scheduler,training_data,logger,boxlength = setup_model(cfg)

dataset:
  cell_len: 1
  input_dir: structures/lj.xyz
  kT: 1.0
  name: mGauss
  ncellx: 2
  ncelly: 2
  ncellz: 2
  nparticles: 20
  rho: None
device: cuda:0
flow:
  hidden_dim: 324
  nlayers: 6
  nsplines: 10
  type: NSF_CL
output:
  model_dir: saved_models/
  pos_dir: ./gauss_positions_during_training.xyz
prior:
  alpha: 100
  lattice_dir: structures/ref.xyz
  type: normal
train_parameters:
  batch_size: 30
  learning_rate: 0.0001
  lr_scheduler_gamma: 0.999
  max_epochs: 4000
  output_freq: 100
B: 1.0


/home/sherryli/xsli/softwares/anaconda3/envs/sherry/lib/python3.9/site-packages/numpy/core/fromnumeric.py:43: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  result = getattr(asarray(obj), method)(*args, **kwds)


RuntimeError: Trying to create tensor with negative dimension -493: [-493, 324]

In [ ]:
train(cfg,model,optimizer,scheduler,training_data,logger,boxlength)